In [ ]:
"""!pip install gym_super_mario_bros==7.4.0
!pip install gym==0.26.2
!pip install nes_py==8.2.1"""

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import torch,gym
from dataclasses import dataclass
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from gym.wrappers import GrayScaleObservation,FrameStack,ResizeObservation
from gym.vector import AsyncVectorEnv
import numpy as np

@dataclass(frozen=False)
class hypers:
    seed : int = 42
    lambda_ : float = .99
    gamma : float = .99
    epsilon : float = .2
    lr : float = 1e-4
    critic_coeff : float = 5e-1
    policy_ceoff : int = 3e3
    entropy_coeff : float = 1e-1
    skip_frame : int = 4
    num_stack : int  = 4
    obs_shape : tuple[int,int] = (100,100) # observation shape
    num_env : int = 8
    num_game : int = 1_000
    batchsize : int = 512 
    minibatch : int = 16
    optim_steps : int = 10
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

configs = hypers()

torch.manual_seed(configs.seed)
np.random.seed(configs.seed)
torch.cuda.manual_seed(configs.seed)

In [ ]:
class CustomEnv(gym.Wrapper): 
        def __init__(self,env,skip):
            super().__init__(env)
            self.skip = skip
            self.score = 0
            
        def step(self, action):
            total_reward = 0  
            for _ in range(self.skip):
                obs,reward,done,truncared,info = self.env.step(action)
                total_reward += reward
                if done:
                    obs,info = self.reset()
                    return obs,(total_reward/10.),done,truncared,info
            return obs,(total_reward/10.),done,truncared,info

        def reset(self, **kwargs):
            self.score = 0
            obs,info = self.env.reset(**kwargs)
            return obs,info

def make_env():
    def env():
        x = gym_super_mario_bros.make("SuperMarioBros-v0", apply_api_compatibility=True)
        x = JoypadSpace(x, SIMPLE_MOVEMENT)
        x = ResizeObservation(x, configs.obs_shape)
        x = CustomEnv(x, configs.skip_frame)
        x = GrayScaleObservation(x, keep_dim=True)
        x = FrameStack(x, configs.num_stack) 
        return x
    return AsyncVectorEnv([env for _ in range(configs.num_env)]) 

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class network(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.LazyConv2d(32,1,1,0)
        self.conv2 = nn.LazyConv2d(32,3,2,2)
        self.conv3 = nn.LazyConv2d(32,3,2,2)
        self.conv4 = nn.LazyConv2d(32,3,2,2)
        self.output = nn.LazyLinear(512)
        self.policy_head = nn.LazyLinear(7)
        self.value_head = nn.LazyLinear(1)
        self.optim = torch.optim.Adam(self.parameters(),lr=configs.lr)
        
    def forward(self,x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x) 
        x = torch.flatten(x,start_dim=1)  
        x = F.relu(self.output(x))
        policy_output = self.policy_head(x)
        value_output = self.value_head(x)
        return F.softmax(policy_output,-1),value_output.squeeze(-1)

def init_weights(layer):
    if isinstance(layer,(nn.Conv2d,nn.Linear)):
        nn.init.orthogonal_(layer.weight)
        nn.init.constant_(layer.bias,0.0)
     
model = network().to(configs.device)
model(torch.rand((1,configs.num_stack,*(configs.obs_shape)),device=configs.device).float())
model.apply(init_weights)
model.compile(mode="reduce-overhead")

In [ ]:
from torch.distributions import Categorical
from collections import deque
from torch import Tensor
import sys

class Memory:
    def __init__(self,env:AsyncVectorEnv):
        N = configs.num_env
        B = configs.batchsize = 10
        self.state = torch.empty((B,N,4,100,100),device=configs.device,dtype=torch.half)
        self.rewards = torch.empty((B,N),device=configs.device,dtype=torch.float32) 
        self.values = torch.empty((B,N),device=configs.device,dtype=torch.float32)
        self.prob = torch.empty((B,N),device=configs.device,dtype=torch.float32) 
        self.action = torch.empty((B,N),device=configs.device,dtype=torch.float32)
        self.dones = torch.empty((B,N),device=configs.device,dtype=torch.bool) 
        self.dist_prob = torch.empty((B,N,7),device=configs.device,dtype=torch.float32) 
        self.advantages = torch.empty((B,N),device=configs.device,dtype=torch.float32) 

        self.env = env
        self.gamma = configs.gamma
        self._lambda_ = configs.lambda_
        self.data = []
        self.pointer = 0
        self.finished_reward = deque(maxlen=30)
        self.log_total_steps = deque(maxlen=30)
        self.episode_reward = torch.empty(self.env.num_envs).float()
        self.total_steps = torch.empty(configs.num_env).float()
        self.transf_obs = lambda x : torch.from_numpy(np.array(x)).squeeze(-1).to(configs.device).float() / 255.
    
    @torch.no_grad()
    def rollout(self,batchsize,network:network):
        self.pointer = 0 
        self._observation,_ = self.env.reset()
        torch.compiler.cudagraph_mark_step_begin() # rollout n>1 times calls compute advantage fn n > 1 times
        for n in range(batchsize):
            self._observation = self.transf_obs(self._observation)
            policy_output, value = network(self._observation)
            distribution = Categorical(policy_output)
            action = distribution.sample()
            prob = distribution.log_prob(action)
            state,reward,done,_,_ = self.env.step(action.cpu().numpy())
            for i in range(self.env.num_envs): # tracking episode rewards and total steps
                self.episode_reward[i] += reward[i] 
                self.total_steps[i] += 1
                if done[i]:
                    self.finished_reward.append(self.episode_reward[i])
                    self.log_total_steps.append(self.total_steps[i])
                    self.episode_reward[i] = 0
                    self.total_steps[i] = 0
            
            self.state[n].copy_(self._observation)
            self.rewards[n].copy_(torch.as_tensor(reward))
            self.values[n].copy_(value)
            self.prob[n].copy_(prob)
            self.action[n].copy_(action)
            self.dones[n].copy_(torch.as_tensor(done))
            self.dist_prob[n].copy_(distribution.probs)
            self._observation = state 

        self.compute_advantage(network,self.rewards,self.values,self.dones)
 
    @torch.compile(mode="reduce-overhead",fullgraph=True)
    def compute_advantage(self,network,rewards:Tensor,values:Tensor,dones:Tensor):
        next_state = self.transf_obs(self._observation)
        _,next_value = network(next_state)
        _values = torch.cat([values,next_value.unsqueeze(0)])
        gae = torch.zeros_like(rewards[0], device=configs.device)
        td = rewards.clone().add_(self.gamma * _values[1:] * (1 - dones)).sub_(_values[:-1])
        for n in reversed(range(len(rewards))): 
            gae.mul_(self._lambda_ * self.gamma * (1-dones[n])).add_(td[n])
            self.advantages[n].copy_(gae)
         
    def sample(self,minibatch): 
        batch = torch.randperm(minibatch)
        #output = self.data[self.pointer:self.pointer+number]
        #self.pointer+=number
        #states,_,values,logProb,actions,_,old_logits,advantages = map(torch.stack,zip(*output))
        #return (states.flatten(0,1),actions,values.squeeze().view(-1),logProb,advantages.view(-1),old_logits)
    
    def traj_reward(self):
        return list(map(torch.tensor,(self.finished_reward,self.log_total_steps)))

Memory(make_env()).rollout(10,network())


In [ ]:
from torch.utils.tensorboard import SummaryWriter
from torch.distributions.kl import kl_divergence
from tqdm import tqdm

class agent:
    def __init__(self,):
        self.env = make_env()
        self.network = model
        self.memory = Memory(self.env)
        self.writter = SummaryWriter("./")
        self.scaler = torch.GradScaler()

    def save(self,k):
        checkpoint = {
            "model_state" : self.network.state_dict(),
            "optim_state" : self.network.optim.state_dict(),
        }
        torch.save(checkpoint,f"./mario{k}")
    
    @torch.compile(mode="reduce-overhead",fullgraph=True)
    def update(self,states,minibatch,cat_old_logits,actions,old_log_prob,advantages,vtarget):
        with torch.autocast(device_type=configs.device,dtype=torch.float16):
            p_output,self.new_values = self.network(states)
            policy_output = p_output.view(minibatch,configs.num_env,7) # 7 = action space for a single env
            self.dist = Categorical(probs=policy_output)  
            self.kl = kl_divergence(cat_old_logits,self.dist).mean() 
            new_log_prob = self.dist.log_prob(actions)
            ratio = torch.exp(new_log_prob - old_log_prob).view(-1) # ratio at step 1 should be 1
            prox1 = ratio * advantages
            prox2 = torch.clamp(ratio ,1-configs.epsilon,1+configs.epsilon) * advantages
            self.loss_policy = -torch.mean(torch.min(prox1,prox2))  
            self.loss_critic = F.smooth_l1_loss(self.new_values.squeeze(),vtarget) * configs.critic_coeff
            entropy = self.dist.entropy().mean() * configs.entropy_coeff
            self.total_loss = self.loss_policy + self.loss_critic - entropy 

    def train(self,train,num_game,batchsize,minibatch,optim_steps):
        if train:
            for traj in tqdm(range(num_game),total=num_game):
                self.memory.rollout(batchsize,self.network)
                for _ in range(batchsize//minibatch):
                    states,actions,old_values,old_log_prob,advantages,old_logits = self.memory.sample(minibatch) 
                    vtarget = (advantages + old_values)
                    explained_variance = 1.0 - (torch.var(vtarget - old_values) / torch.var(vtarget + 1e-10))
                    cat_old_logits = Categorical(probs=old_logits)
                    for _ in range(optim_steps): 
                        self.network.optim.zero_grad(set_to_none=True)
                        self.update(states,minibatch,cat_old_logits,actions,old_log_prob,advantages,vtarget)
                        self.scaler.scale(self.total_loss).backward()
                        self.scaler.unscale_(self.network.optim)
                        nn.utils.clip_grad_norm_(self.network.parameters(), 0.5)
                        self.scaler.step(self.network.optim)
                        self.scaler.update()
                
                if traj%5==0: # log every five iter...
                    self.writter.add_scalar("Policy/entropy",self.dist.entropy().mean(),traj)
                    self.writter.add_scalar("Policy/loss policy",self.loss_policy,traj)
                    self.writter.add_scalar("Policy/KL",self.kl,traj)
                    self.writter.add_scalar("Value/values",self.new_values.detach().mean(),traj)
                    self.writter.add_scalar("Value/vtarget",vtarget.mean(),traj)
                    self.writter.add_scalar("Value/value loss",self.loss_critic,traj) 
                    self.writter.add_scalar("Value/Explained variance",explained_variance,traj)
                    self.writter.add_scalar("main/total loss",self.total_loss,traj)
                    self.writter.add_scalar("main/epi rewards",self.memory.traj_reward()[0].mean(),traj) 
                    self.writter.add_scalar("main/total steps",self.memory.traj_reward()[1].mean(),traj)

            if traj % 50 == 0 :  
                self.save(traj)
        
        self.save("Final")
    
agent().train(train = True,
    num_game=configs.num_game,
    batchsize=configs.batchsize,
    minibatch=configs.minibatch,
    optim_steps=configs.optim_steps
    ) 